# Acne Segmentation & Severity Classification
**Dataset:** ACNE04-v2 | **Model:** Hybrid ResNet-50 + Dilated Decoder + AMFM

---
## Cara Pakai
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Isi `GITHUB_URL` di cell SETUP di bawah
3. **Runtime → Run all** — selesai
---

## ⚙️ SETUP — Isi GitHub URL kamu, lalu Run All

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  GANTI URL INI DENGAN REPO GITHUB KAMU              ║
# ╚══════════════════════════════════════════════════════╝
GITHUB_URL = 'https://github.com/Pandeigede2803/acne_segmentation_github.git'

# ── Jangan ubah di bawah ini ───────────────────────────
import os, sys, torch
from pathlib import Path

REPO_DIR    = '/content/acne-seg'
OUTPUT_ROOT = '/content/drive/MyDrive/acne_project/output'

# Cek GPU
assert torch.cuda.is_available(), '\n⚠️  GPU belum aktif!\nRuntime → Change runtime type → T4 GPU → Save'
print('✅ GPU :', torch.cuda.get_device_name(0))
print('   VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

# Clone atau pull repo (scripts .py)
if os.path.exists(REPO_DIR):
    !git -C "{REPO_DIR}" pull --quiet
    print('✅ Scripts diupdate dari GitHub')
else:
    !git clone --quiet "{GITHUB_URL}" "{REPO_DIR}"
    print('✅ Scripts berhasil diload dari GitHub')

SCRIPTS_DIR = Path(REPO_DIR) / 'scripts'
for p in [str(SCRIPTS_DIR), str(SCRIPTS_DIR / 'code_dari_claude')]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('   Scripts:', [f.name for f in SCRIPTS_DIR.glob('*.py')])

---
## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive terhubung')

---
## 2. Install Dependencies

In [ ]:
# ── Download dataset dari Google Drive (public) ───────────
import gdown, os
from pathlib import Path

DOWNLOAD_DIR = Path('/content/acne_images')
TAR_PATH     = DOWNLOAD_DIR / 'ACNE04' / 'Classification.tar'
EXTRACT_DIR  = DOWNLOAD_DIR / 'extracted'

# Download hanya jika belum ada
if not TAR_PATH.exists():
    gdown.download_folder(
        'https://drive.google.com/drive/folders/18yJcHXhzOv7H89t-Lda6phheAicLqMuZ',
        output=str(DOWNLOAD_DIR),
        quiet=False
    )

# Ekstrak tar jika belum diekstrak
if not EXTRACT_DIR.exists():
    EXTRACT_DIR.mkdir(parents=True)
    print('Mengekstrak Classification.tar...')
    !tar -xf "{TAR_PATH}" -C "{EXTRACT_DIR}"
    print('✅ Ekstrak selesai')
else:
    print('ℹ️  Sudah diekstrak sebelumnya — skip.')

# Cari subfolder yang berisi gambar .jpg
jpg_dirs = [p.parent for p in EXTRACT_DIR.rglob('*.jpg')]
if jpg_dirs:
    IMAGE_DIR = sorted(set(jpg_dirs), key=lambda p: -len(list(p.glob('*.jpg'))))[0]
else:
    IMAGE_DIR = EXTRACT_DIR

n = len(list(IMAGE_DIR.glob('*.jpg')))
print(f'✅ IMAGE_DIR: {IMAGE_DIR}')
print(f'   Total gambar: {n}')

# ── OPTION B: Kaggle ─────────────────────────────────────
# Uncomment jika pakai Kaggle, comment cell Option A di atas

# import kagglehub
# from pathlib import Path
# KAGGLE_PATH = kagglehub.dataset_download('karmagames/acne04-v2')
# IMAGE_DIR   = Path(KAGGLE_PATH)
# print('✅ Dataset:', IMAGE_DIR)

In [ ]:
# ── OPTION A: Kaggle (Rekomendasi) ───────────────────────
import kagglehub
KAGGLE_PATH = kagglehub.dataset_download('karmagames/acne04-v2')
IMAGE_DIR   = Path(KAGGLE_PATH)
print('✅ Dataset:', IMAGE_DIR)
!ls "{IMAGE_DIR}" | head -5

In [ ]:
# ── OPTION B: Google Drive ────────────────────────────────
# Uncomment cell ini jika pakai Google Drive
# Comment cell Option A di atas terlebih dahulu

# import gdown
# gdown.download_folder(
#     'https://drive.google.com/drive/folders/18yJcHXhzOv7H89t-Lda6phheAicLqMuZ',
#     output='/content/acne_images/',
#     quiet=False
# )
# IMAGE_DIR = Path('/content/acne_images')
# print('✅ Dataset:', IMAGE_DIR)

---
## 4. Download Anotasi v2

In [ ]:
import json
ANNOT_REPO = '/content/acne04v2'

if os.path.exists(ANNOT_REPO):
    !git -C "{ANNOT_REPO}" pull --quiet
else:
    !git clone --quiet https://github.com/AIpourlapeau/acne04v2 "{ANNOT_REPO}"

ANNOT_JSON = Path(f'{ANNOT_REPO}/Acne04-v2_annotations.json')
with open(ANNOT_JSON) as f:
    d = json.load(f)
print(f'✅ Anotasi: {len(d.get("images",[]))} gambar | {len(d.get("annotations",[]))} lesi')

---
## 5. Konfigurasi Path Output

In [ ]:
OUTPUT_DIR = Path(OUTPUT_ROOT)
MASK_DIR   = OUTPUT_DIR / 'refined_masks_output'
TRAIN_DIR  = OUTPUT_DIR / 'training_runs'
INFER_DIR  = OUTPUT_DIR / 'predictions'

for d in [MASK_DIR, TRAIN_DIR, INFER_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE           = 'cuda' if torch.cuda.is_available() else 'cpu'
REFINED_MANIFEST = MASK_DIR / 'refined_manifest.csv'
OUTPUT_RUN       = TRAIN_DIR / 'proposal_hybrid_v1'

errors = []
if not SCRIPTS_DIR.exists(): errors.append(f'❌ Scripts: {SCRIPTS_DIR}')
if not IMAGE_DIR.exists():   errors.append(f'❌ Images : {IMAGE_DIR}')
if not ANNOT_JSON.exists():  errors.append(f'❌ Annot  : {ANNOT_JSON}')

if errors:
    for e in errors: print(e)
else:
    n = len(list(IMAGE_DIR.rglob('*.jpg')))
    print(f'✅ Siap | {n} gambar | device: {DEVICE}')
    print(f'   Output → {OUTPUT_DIR}')

---
## 6. Generate Refined Mask

Otomatis skip jika sudah pernah dijalankan sebelumnya.

In [ ]:
import pandas as pd

if REFINED_MANIFEST.exists():
    df = pd.read_csv(REFINED_MANIFEST)
    print(f'ℹ️  Manifest sudah ada ({len(df)} baris) — skip.')
    print(df['split'].value_counts().to_string())
else:
    !python "{SCRIPTS_DIR}/refine_masks_color.py" \
      --image-dir       "{IMAGE_DIR}" \
      --annotations-json "{ANNOT_JSON}" \
      --output-dir      "{MASK_DIR}" \
      --apply-preprocessing \
      --max-images 0

    if REFINED_MANIFEST.exists():
        df = pd.read_csv(REFINED_MANIFEST)
        print(f'\n✅ Manifest dibuat: {len(df)} baris')
        print(df['split'].value_counts().to_string())
    else:
        print('❌ Gagal generate mask. Cek error di atas.')

---
## 7. Training

Estimasi: **2–3 jam** di T4 GPU.

Jika sesi Colab putus → uncomment cell **RESUME** di bawah, lalu jalankan ulang dari cell SETUP.

In [ ]:
# ── TRAINING BARU ─────────────────────────────────────────
!python "{SCRIPTS_DIR}/train_proposal_hybrid_resnet50_malds.py" \
  --manifest        "{REFINED_MANIFEST}" \
  --output-dir      "{OUTPUT_RUN}" \
  --image-size      320 \
  --batch-size      16 \
  --phase1-epochs   30 \
  --phase2-epochs   90 \
  --device          cuda \
  --pretrained \
  --weighted-sampler \
  --num-workers     4 \
  --seed            42

In [ ]:
# ── RESUME (jika sesi putus) ──────────────────────────────
# Uncomment semua, comment cell TRAINING BARU di atas

# !python "{SCRIPTS_DIR}/train_proposal_hybrid_resnet50_malds.py" \
#   --manifest        "{REFINED_MANIFEST}" \
#   --output-dir      "{OUTPUT_RUN}" \
#   --image-size      320 \
#   --batch-size      16 \
#   --phase1-epochs   30 \
#   --phase2-epochs   90 \
#   --device          cuda \
#   --pretrained \
#   --weighted-sampler \
#   --num-workers     4 \
#   --resume

---
## 8. Monitor Progress

In [ ]:
import json, matplotlib.pyplot as plt

metrics_path = OUTPUT_RUN / 'metrics.json'
if not metrics_path.exists():
    print('metrics.json belum ada — training belum mulai.')
else:
    with open(metrics_path) as f:
        data = json.load(f)
    h  = data.get('history', [])
    ep = [x['epoch'] for x in h]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, key, title in zip(axes, ['dice','acc','loss'], ['Dice','Accuracy','Loss']):
        ax.plot(ep, [x.get(f'train_{key}', 0) for x in h], label='Train')
        ax.plot(ep, [x.get(f'val_{key}',   0) for x in h], label='Val')
        ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)

    last = h[-1]
    plt.suptitle(
        f"Epoch {last['epoch']} | {last.get('phase','?')} | "
        f"Val Dice {last.get('val_dice',0):.4f} | Val Acc {last.get('val_acc',0):.4f}"
    )
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / 'training_curves.png'), dpi=100)
    plt.show()

---
## 9. Hasil Test Metrics

In [ ]:
import json
metrics_file = OUTPUT_RUN / 'metrics.json'
if not metrics_file.exists():
    print('Training belum selesai.')
else:
    with open(metrics_file) as f:
        data = json.load(f)
    tm = data.get('test_metrics', {})
    print('=' * 38)
    print('        HASIL TEST SET')
    print('=' * 38)
    print(f"  Accuracy    : {tm.get('acc',   0):.4f}")
    print(f"  Kappa       : {tm.get('kappa', 0):.4f}")
    print(f"  Dice Score  : {tm.get('dice',  0):.4f}")
    print(f"  IoU         : {tm.get('iou',   0):.4f}")
    print('-' * 38)
    print(f"  Seg Loss    : {tm.get('seg_loss', 0):.4f}")
    print(f"  Cls Loss    : {tm.get('cls_loss', 0):.4f}")
    print('=' * 38)
    print(f"  Best Val Loss: {data.get('best_val_loss',0):.4f}")

---
## 10. Inferensi & Visualisasi

In [ ]:
CHECKPOINT = OUTPUT_RUN / 'best_model.pt'
INFER_OUT  = INFER_DIR  / 'proposal_hybrid_v1'

if not CHECKPOINT.exists():
    print('❌ best_model.pt tidak ada. Training harus selesai dulu.')
else:
    !python "{SCRIPTS_DIR}/infer_testing_images.py" \
      --manifest    "{REFINED_MANIFEST}" \
      --checkpoint  "{CHECKPOINT}" \
      --output-dir  "{INFER_OUT}" \
      --split       test \
      --image-size  320 \
      --threshold   0.5 \
      --device      cuda

    import matplotlib.pyplot as plt
    from PIL import Image as PILImage
    pred_files = sorted(Path(INFER_OUT).glob('*.png'))[:8]
    if pred_files:
        fig, axes = plt.subplots(2, 4, figsize=(18, 8))
        for ax, pf in zip(axes.flatten(), pred_files):
            ax.imshow(PILImage.open(pf))
            ax.set_title(pf.stem[:20], fontsize=8)
            ax.axis('off')
        plt.suptitle('Predicted Segmentation Masks — Test Set')
        plt.tight_layout()
        plt.savefig(str(OUTPUT_DIR / 'predicted_masks.png'), dpi=100)
        plt.show()